<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Pandas II: Exploring, Cleaning, and Joining Data

### CP101: Introduction to Urban Data Analytics

**Week 6, Monday** · Pandas II

**Not graded, not submitted.** Run cells in order with `Shift + Enter`. Complete each **Try It Out**; the Otter cells offer optional feedback on your results. Chaining methods or writing separate steps both work; keep the requested variable names for later cells and checks. Discuss interpretation with a partner. A few expandable explanations and a worked merge are available after you try.

[Course guide](https://ucb-cp101.github.io/python-resources/tutorials/basic/pandas-data-explore/) · [pandas cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_W06_S1_pandas_exploration.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Learning Objectives

<hr style="border: 1px solid #fdb515;" />

By the end, you should be able to:

1. Explain what a column's dtype describes
2. Explain why a dataset metadata is essential to cleaning and interpreting a dataset.
3. Evaluate record-level data quality, including repeated rows and the results of a type conversion.
4. Use `groupby()` to aggregate records at the unit of analysis a question requires.
5. Join two datasets on a shared key and evaluate the result of the join.

The optional extension applies the same reasoning to a subset defined by time.

### The file you are working with

You are working with `apartment_cost_list_20260927.csv`, a community-created NYC Open Data view of **DOB job application filings**. DOB is New York City's Department of Buildings. Before most construction or renovation work, an owner or contractor files a **job application** describing the proposed work, and DOB later issues a **permit** authorizing it to go ahead. Every row comes from that paperwork. A row is not evidence that the work was finished, or that anyone inspected or surveyed a building. [NYC explains applications and permits](https://www.nyc.gov/site/buildings/dob/building-applications-permits.page).

A **view** is a saved presentation of a larger dataset that can select rows and columns. This one is named **apartment cost list**, and the name does not match the contents. 

**We cannot infer the population/objective from the dataset title.**

“**apartment cost list**” is a title chosen by the public user who created this view. The metadata does not establish that every record represents an apartment.

- It is not a list of what apartments sell or rent for.
- `Initial Cost` is the applicant's estimate of what the proposed work will cost, recorded when the application was filed. It is not the value of the building, not what the building cost to construct, and not the final amount spent.
- The view's selection rules are not documented, so it is not a complete record of apartment work in the city.

One row is one filing record, with a building ID, several dates, and an estimated cost for the proposed work.

### In this analysis, we want to answer:

1. **Which boroughs have the most records in this file, and how do their median estimated costs compare?**
2. **How do median estimated costs compare across job types (`A1`, `A2`, and `A3`)?**
3. **After adding building information, which construction decades appear most often in these records, and how do their median estimated costs compare?**
4. **Optional time comparison: how do recorded counts in 2019–2021 compare with nearby years?** Use the fully-permitted date, not a filing or completion date.

To answer these questions, work through the six-step checklist: inspect the columns, interpret missing values and repeated rows, clean names, and convert dates and costs. These checks help us decide which comparisons the recorded values support.

The borough and job-type comparisons use columns already in this file. The construction-decade comparison needs the building's reported construction year, so we will join a second NYC table using the building ID. We will check unmatched rows and unknown years before summarizing by decade. The optional extension uses the cleaned dates to examine recorded counts over time.

Here, **activity means rows recorded in this selected file**. The counts do not measure all construction, unique completed projects, or all buildings in a borough.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Data Exploration Checklist

<hr style="border: 1px solid #fdb515;" />

Use this checklist as a roadmap for the six steps below. For each item, point to a result or explain a decision rather than naming a pandas method.

**1. Load and Inspect**

- Check dimensions (rows × columns).
- View the first few rows and examine column names.
- Have a clear picture of what one row represents and inspect the identifiers.

**2. Understand Data Types**

- Check the dtype of each column.
- Distinguish quantities from identifiers, dates, and text.
- Check whether each column have a correct `dtype` based on its content?
- Inspect ranges and flag representations that need conversion.

**3. Check Data Quality**

- Count missing values and consult metadata about blanks.
- Check which kinds of cases appear in the file.
- Detect repeated rows and explain whether deletion is justified.

**4. Clean Column Names**

- Standardize case, spaces, and special characters.
- Confirm that the cleaned names are unique before assigning them.
- Create a working copy while preserving the source values.

**5. Clean Data**

- Convert date-like text and currency strings to appropriate dtypes.
- Check whether conversion changed values or introduced missingness.
- Inspect the converted values before using them in calculations.

**6. Validate**

- Summarize at the unit needed for the question.
- Join building attributes; check key uniqueness, row counts, and matches.
- Use the added measurement, distinguish records from buildings, and explain the limits.

The optional practice extends validation to failed joins, geographic edge cases, and a time-analysis subset.


### Data and setup

Both CSVs are already in `data/raw/`; no download, API key, or GIS library is needed.

| File | One row represents | Contents |
|---|---|---|
| `apartment_cost_list_20260927.csv` | One visible DOB record; not necessarily a unique job | Building ID, dates, estimated job cost, and other filing fields |
| `nyc_building_attributes_20260927.csv` | One building ID in this extract | Building ID and reported construction year from NYC's building catalog |

The second CSV is a small attribute-only extract from the [official NYC building dataset](https://data.cityofnewyork.us/City-Government/BUILDING/5zhs-2jue). It contains the buildings found when we requested the DOB file's building IDs. It has no map geometry. Both sources are local snapshots, not live feeds.

**Metadata** means documentation about the data, including field definitions and how the file was produced. The building extract was retrieved from NYC Open Data on September 27, 2026, and keeps two columns: the building ID and the reported construction year, with no geometry and no estimated or filled-in years. Keep source files unchanged and work in memory. The course environment already includes pandas.


In [ ]:
!pip install "geopandas[all]"

In [ ]:
import pandas as pd

# A version number helps reproduce behavior when troubleshooting.
print("pandas:", pd.__version__)


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Step 1: Load and Inspect

<hr style="border: 1px solid #fdb515;" />

First identify what a row represents, then inspect the fields.


In [ ]:
# Load the data
permits_df = pd.read_csv('data/raw/apartment_cost_list_20260927.csv')

# Check dimensions
print(f"Shape: {permits_df.shape[0]:,} rows × {permits_df.shape[1]} columns")


In [ ]:
# View first few rows
permits_df.head()


### What does one row represent?

A row is one record in this **derived view** of DOB Job Application Filings.

`Job #` identifies a job filing, but it is **not unique in this file**. The parent DOB data also contain identifiers such as `Doc #` and `JOB_S1_NO` that this view omits.

So do not assume:

**one row = one unique job = one building**

For example, one building could have separate plumbing and renovation job filings. Its building ID would repeat even if the job IDs differed. Repeated job IDs within this view are a separate issue.

Let’s check the row count against the number of unique job numbers.


In [ ]:
print(f"Rows: {len(permits_df):,}")
print(f"Unique Job #: {permits_df['Job #'].nunique():,}")


In [ ]:
# List all column names
permits_df.columns.tolist()


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Step 2: Understand Data Types

<hr style="border: 1px solid #fdb515;" />

`info()` reports each column's dtype and its non-null count. Before running it, predict which fields contain **quantities**, **identifiers**, and **dates**.


In [ ]:
permits_df.info(memory_usage='deep')


**What do we notice?**

- **`Fully Permitted`** is `object`: CSV dates are normally read as text unless we explicitly parse them as dates.
- **`Initial Cost`** is `object`: values contain `$`, so pandas cannot treat them as numbers yet.
- **`Job #`**, **`Block`**, **`Lot`**, **`Bin #`**, and **`Community - Board`** are stored as integers because they contain digits, but they are **identifiers or geographic codes**, not quantities.

`read_csv()` infers dtypes from the values it sees. In the NYC metadata, `Job #`, `Block`, `Lot`, `Bin #`, and `Community - Board` are defined as **Text**, even though pandas infers `int64` for them in this CSV. Missing or mixed values can also affect the inferred dtype. For example, a traditional integer column containing `NaN` may be represented as `float64`, while mixed numeric and text values may be read as `object`.

### Numerical summaries need meaningful quantities

`describe()` summarizes numeric columns. The 25%, 50%, and 75% entries are quartiles; 50% is the median. Which numeric columns should **not** have their means interpreted?


In [ ]:
permits_df.describe()
# permits_df.describe(include="all")


### Compare your interpretation

Which means would you avoid reporting? What do the zero quartiles tell you, and what do they leave out?

<details>
<summary>Discuss first, then compare</summary>

`Job #`, `Block`, `Lot`, `Bin #`, and `Community - Board` are identifiers or geographic codes; their means have no useful interpretation here. `Proposed Zoning Sqft` and `Enlargement SQ Footage` are quantities in square feet. Their first quartile, median, and third quartile are all zero, so the summary hides variation among the nonzero records.

</details>

Next, calculate how common zero is in the two square-footage fields.


In [ ]:
# True counts as 1 and False as 0, so the mean is the proportion of zeros.
zero_flags = permits_df[["Proposed Zoning Sqft", "Enlargement SQ Footage"]] == 0
zero_share = zero_flags.mean()
# Multiply by 100 to convert proportions to percentages; round to one decimal place.
(zero_share * 100).round(1)


About **97.2%** of `Proposed Zoning Sqft` and **98.1%** of `Enlargement SQ Footage` are zero in this file.

Zero is **not automatically missing data or an error**. It can be a true zero, a value that does not apply to this filing, or an administrative default. The metadata does not say which, so we have to investigate.

**The technique:** when documentation does not tell you what a value means, test it against another column in the same file that ought to agree with it. Agreement supports one reading. Disagreement rules one out.

`Horizontal Enlrgmt` and `Vertical Enlrgmt` record whether a filing enlarges the building outward or upward. If `Enlargement SQ Footage` measures an enlargement, then it should be zero exactly when neither flag is set. That is a prediction we can check.

In [ ]:
enlargement_flagged = (
    permits_df["Horizontal Enlrgmt"].eq("Y")
    | permits_df["Vertical Enlrgmt"].eq("Y")
)

# Rows down: is the filing flagged as an enlargement? Columns across: is the footage zero?
pd.crosstab(
    enlargement_flagged.rename("enlargement flagged"),
    (permits_df["Enlargement SQ Footage"] == 0).rename("enlargement sqft is 0"),
)

The prediction holds. Every unflagged row records zero, and all but two flagged rows record a positive figure. For this column, zero means the filing does not enlarge the building. It is a measurement, not a gap.

Now test `Proposed Zoning Sqft` the same way. It has no companion flag, so use the enlargement figure instead: a filing that adds floor area cannot also propose zero square feet. If such rows exist, zero cannot mean "zero square feet" throughout this column.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Investigate: Does zero mean the same thing in both columns?

Find the filings that contradict the second reading: `Enlargement SQ Footage` greater than zero **and** `Proposed Zoning Sqft` equal to zero.

Build the Boolean mask `zoning_contradictions`, then count it into `num_contradictions`.

**Hint:** combine two comparisons with `&`, putting each one in parentheses. Summing a Boolean mask counts its `True` entries.

In [ ]:
zoning_contradictions = ...
num_contradictions = ...

print("Filings that add floor area but report 0 proposed zoning sqft:", num_contradictions)

In [ ]:
grader.check("ca06_zeros")

These rows record an enlargement of up to 29,687 sq ft while reporting zero proposed zoning square footage. A filing that adds floor area does not propose zero square feet, so here the zero stands for something else: not reported, not applicable to the filing type, or a system default. This file does not tell us which.

<details>
<summary>What follows from that</summary>

The two columns need different handling. Read the `Enlargement SQ Footage` zeros as real measurements, since the flags corroborate them. Treat the `Proposed Zoning Sqft` zeros as unverified, and leave them out of a calculation rather than averaging them in as if they were measured. Neither column should be rewritten: we recorded what we found and carried the uncertainty forward.

Notice that `describe()` showed both columns as a pile of zeros. The difference only appeared once we compared columns against each other.

</details>

**Hand - DD9: Summaries of Data:** summaries discard detail and can hide important structure in the underlying distribution. The Hand/DD labels connect these examples to David Hand's *Dark Data* ([taxonomy overview](https://imstat.org/2019/12/15/xl-files-time-travel-and-dark-data/)).

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Step 3: Check Data Quality

<hr style="border: 1px solid #fdb515;" />

Read missing values, composition, and repeated rows as evidence about the source. Each check raises a question; it does not automatically tell us to replace or delete a value.


### Count blanks, then consult metadata

Does a blank cell necessarily mean missing information? First inspect what pandas reports.


In [ ]:
(
    permits_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head()
)


### The largest "missing" column is not actually missing information

The output puts `Curb Cut` at the top: almost every row is blank. Here, a [curb cut](https://www.nyc.gov/site/buildings/property-or-business-owner/curb-cuts.page) is lowered curb/sidewalk access for a vehicle to enter a driveway or similar space.

But NYC metadata defines the field as:

> **X = Yes, Blank = No**

So pandas reads those blanks as `NaN`, while the source uses the blank itself to encode **No**.

For `Horizontal Enlrgmt` and `Vertical Enlrgmt`, the metadata does not give us an equivalent blank-value rule, so we should **not invent one**.

`Fully Permitted` is different again: a blank means there is no recorded fully-permitted date in that field. We will investigate that later.

[NYC Open Data metadata](https://data.cityofnewyork.us/Housing-Development/apartment-cost-list/s4p6-zx84)

**Hand - DD8: Definitions of Data:** the same visual blank can mean different things depending on how the source defines the field.


In [ ]:
# Include NaN so we can see how pandas reads the blank values
(
    permits_df["Curb Cut"]
    .value_counts(dropna=False)
)


#### What did the code do?

- `.isna()` creates `True` / `False` values for cells pandas considers missing.
- `.sum()` counts the `True` values for each column.
- `.sort_values(ascending=False)` puts the largest counts first.
- `.head()` keeps the first five.

This tells us where pandas sees missing values. **Metadata tells us what those values mean.**


### What kinds of records are in this file?

Before treating the file as representative of NYC DOB activity, check its composition. **Borough** identifies one of NYC’s five major geographic divisions: Manhattan, the Bronx, Brooklyn, Queens, or Staten Island.

`Job Type` records what kind of application was filed. DOB's filing system covers new
construction and demolition as well as changes to existing buildings. An **alteration**
is a change to an existing building. A **Certificate of Occupancy** describes a
building's permitted use and occupancy.

| Code | Meaning in this filing system |
|---|---|
| `NB` | New building |
| `DM` | Demolition or removal |
| `SG` | Sign |
| `A1` | Alteration requiring a new or amended Certificate of Occupancy |
| `A2` | Multiple work types without that certificate change, such as construction and plumbing |
| `A3` | One work type without that certificate change, such as a curb cut or construction fence |

[NYC’s application guide, Section 5](https://www.nyc.gov/assets/buildings/pdf/pw1_userguide.pdf). These categories describe the filing; they do not identify the number of apartments or completed projects.


In [ ]:
# normalize=True returns proportions among records with a nonmissing value.
borough_share = permits_df["Borough"].value_counts(normalize=True)
# Convert proportions to percentages and round to one decimal place.
(borough_share * 100).round(1)


In [ ]:
# normalize=True returns proportions among records with a nonmissing value.
job_type_share = permits_df["Job Type"].value_counts(normalize=True)
# Convert proportions to percentages and round to one decimal place.
(job_type_share * 100).round(1)


In this snapshot:

- about **78%** of rows are in Manhattan
- about **92%** are `A2`
- only `A1`, `A2`, and `A3` job types appear

The percentages count rows in this file. For example, 78% does not mean that 78% of NYC buildings are in Manhattan.

The metadata says this is a **community-created view**, and NYC Open Data warns that such views may contain filters.

So we should describe this as **the records in this view**, not as all NYC DOB job applications.

**Hand - DD3: Choosing Just Some Cases:** conclusions depend on which cases made it into the dataset.


### Investigate repeated rows

`duplicated()` marks a row `True` when all its visible values repeat an earlier row. The first occurrence is `False`. This counts **additional repeated rows**, not every member of a repeated group.

The parent data contain identifiers omitted from this view. Identical visible rows therefore do not establish that the underlying source records are redundant.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 1: Count Repeated Rows

Use the supplied Boolean mask to calculate `num_duplicates` and `pct_duplicates`. Express the percentage on a 0–100 scale; rounding to one decimal place is fine.

No column in this file is unique, and neither is the combination of all 18. The parent data identifies a document within a job, but this view does not carry that field, so two identical rows cannot be distinguished from one row recorded twice. That is why we do not delete them.


**Hint:** summing a Boolean mask counts its `True` entries. After the check, discuss: **Does this result alone justify deleting those rows?**


In [ ]:
repeated_mask = permits_df.duplicated()
num_duplicates = ...
pct_duplicates = ...

print(f"Additional repeated rows: {num_duplicates:,} ({pct_duplicates:.1f}%)")


In [ ]:
grader.check("ca06_duplicates")

#### What did we find?

In this file, **5,947 rows (30.4%)** repeat an earlier row across all 18 visible columns.

That is a property of this view. It is **not enough evidence** to conclude that 5,947 source records are erroneous duplicates.


In [ ]:
# Demonstration only: remove exact repeats across the visible columns.
permits_no_exact_repeats = (
    permits_df
    .drop_duplicates()
)

print(f"Rows before: {len(permits_df):,}")
print(f"Rows after exact-row removal: {len(permits_no_exact_repeats):,}")
print(f"Difference: {len(permits_df) - len(permits_no_exact_repeats):,}")


#### Should we use the deduplicated version?

Not for the analysis below.

We do not have enough information in this view to establish that every repeated visible row is a redundant source record. We therefore preserve all rows and create a working copy.

`drop_duplicates()` is appropriate when the duplication rule is known, for example, when records were accidentally repeated or a valid key defines uniqueness.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Step 4: Clean Column Names

<hr style="border: 1px solid #fdb515;" />

Consistent names make later expressions easier to read. We will give a working copy consistent names while keeping `permits_df` unchanged, so earlier inspection cells can still run.

Read the following chain one line at a time, starting at the top. Parentheses let the expression span lines without a backslash.


In [ ]:
new_names = (
    permits_df.columns
    .str.strip()
    .str.replace(" #", "_num", regex=False)
    .str.replace(" - ", "_", regex=False)
    .str.replace(" ", "_", regex=False)
    .str.lower()
)
new_names


Different source names can collapse to the same cleaned name. Check uniqueness **before** assigning the new names; an assertion stops execution if the condition is false.


In [ ]:
assert new_names.is_unique, "The cleaning produced duplicate column names."
permits_work = permits_df.copy()
permits_work.columns = new_names
permits_work.columns.tolist()


### Keep a working copy

We retain all rows because this view does not supply a defensible deduplication key. `permits_df` keeps the original values **and column names**. `permits_work` has the cleaned names and will hold the converted dates and costs.

For example, use `permits_df["Initial Cost"]` to read the source text and `permits_work["initial_cost"]` for the working column. Recreating the working copy resets its conversions; rerun Step 5 afterward.


In [ ]:
permits_work.shape


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Step 5: Clean Data

<hr style="border: 1px solid #fdb515;" />

Convert representations only after inspecting their meaning. We will check both date parsing and numerical amounts, then inspect the converted values.


### 5.1 Convert dates and check the result

Inspect a reproducible sample before choosing a date format. `%m/%d/%Y` means month/day/four-digit year.


In [ ]:
permits_work["fully_permitted"].sample(5, random_state=42)


In [ ]:
permits_work["permit_date"] = pd.to_datetime(
    permits_work["fully_permitted"],
    format="%m/%d/%Y",
    errors="coerce",
)

permits_work["permit_date"]


**What happened here?**

1. `pd.to_datetime()` parses the text as dates.
2. `format="%m/%d/%Y"` states the expected month/day/year format.
3. `errors="coerce"` tells pandas: convert valid values, and turn values that cannot be converted into missing values (`NaT`, "Not a Time") instead of stopping with an error.

Because coercion can silently create new missing values, we should **validate the conversion** rather than assume it worked.


In [ ]:
# Did any non-missing source value fail to parse?
parse_failed = (
    permits_work["fully_permitted"].notna()
    & permits_work["permit_date"].isna()
)

parse_failed.sum()


The result is **0**: every non-blank `Fully Permitted` value in this file parsed successfully. The existing blanks became `NaT`; the conversion did not create additional missing dates.


### Pause and explain

If a nonblank date were mistyped, how would `errors="coerce"` affect it? Why do we check `parse_failed` afterward?

<details>
<summary>Discuss first, then compare</summary>

An unparseable value becomes `NaT`, the missing value for datetime data. Without the check, we could mistake a parsing failure for a blank in the source. The default, `errors="raise"`, stops on an invalid value; that can be useful when unexpected formats should be investigated immediately. Missing-value markers depend on dtype: pandas also uses `NaN` and `pd.NA` in other column types.

</details>


The `.dt` accessor provides date components, much as `.str` provides string operations. For example, `permits_work["permit_date"].dt.month` extracts the month; missing dates stay missing.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 2: Extract and Count Years

Create `perm_year` from `permit_date`, then count records for each nonmissing year in `permits_per_year`. Use `.dt.year` and `.value_counts()`.

These are **record counts by fully-permitted year**. A fully-permitted date is not a filing date or a construction completion date.


In [ ]:
permits_work["perm_year"] = ...
permits_per_year = ...
permits_per_year


In [ ]:
grader.check("ca06_permyear")

### What can we say about a blank `fully_permitted` value?

In this file, **3,952 rows (20.2%)** have no `Fully Permitted` value.

NYC defines this field as the **date when the job is fully permitted**. We can therefore say that no fully-permitted date is recorded in this field for those rows.

We should **not** call those rows "pending," "not approved," or assume why the date is absent. The reason is not visible in this view.

> **A cell represented as missing by pandas is not automatically evidence that information which should exist is missing from the world.**


### 5.2 Convert and inspect costs

`initial_cost` is an estimated dollar amount for the work covered by a filing. It is neither apartment rent nor the building’s sale price, and it is not a measured final expenditure. Inspect the text, convert it, and check that the conversion preserves the recorded amounts.


In [ ]:
permits_work[["initial_cost"]].head()


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 3: Make Costs Numeric

Create `cost_numeric` by removing `$` and any commas from `permits_df["Initial Cost"]`, then converting to `float`. Read the preserved text in `permits_df`, so this conversion can be rerun after changing the working copy.

**Hint:** chain `.str.replace("$", "", regex=False)`, `.str.replace(",", "", regex=False)`, and `.astype(float)`. `regex=False` treats the character literally.


In [ ]:
cost_numeric = ...
cost_numeric.head()


In [ ]:
grader.check("ca06_costs")

In [ ]:
# Replace the working column only after checking the conversion.
permits_work["initial_cost"] = cost_numeric

# Compare the original text with the converted dollar amounts.
cost_comparison = pd.DataFrame({
    "Original text": permits_df["Initial Cost"],
    "Numeric dollars": permits_work["initial_cost"],
})
cost_comparison.head()


Compare the two columns: the dollar signs and commas should be gone, but the dollar amounts should be unchanged. We can now compare costs by borough and job type, then add building information for another comparison.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Step 6: Validate

<hr style="border: 1px solid #fdb515;" />

Start with the cleaned DOB table: compare **record counts and estimated costs by borough**, then by **job type**. Afterward, join building information to compare **reported construction decades**.

The first two comparisons use columns we already have. The third requires the join because neither table alone contains both construction year and job cost. We will preserve every DOB row, check the matches, then summarize the rows with a usable construction year.


### 6.1 Compare boroughs, then job types

`.groupby("borough")` puts rows with the same borough into a group. `.agg()` calculates summaries for each group. In the code below:

- `record_count=("initial_cost", "size")` counts rows and names the result `record_count`.
- `median_initial_cost=("initial_cost", "median")` finds the median estimated cost in dollars.
- `.sort_values()` puts the borough with the most records first.

The median is the middle cost after sorting, or the average of the middle two when the group has an even number of rows. Each recorded row contributes once, including repeated visible rows.

In [ ]:
borough_summary = (
    permits_work.groupby("borough")
    .agg(
        record_count=("initial_cost", "size"),
        median_initial_cost=("initial_cost", "median"),
    )
    .sort_values("record_count", ascending=False)
)
borough_summary


Now change the grouping column to `job_type`. The calculations stay the same, but the question is now **what kind of alteration was filed**, not **where the filing is located**.

In [ ]:
job_type_summary = (
    permits_work.groupby("job_type")
    .agg(
        record_count=("initial_cost", "size"),
        median_initial_cost=("initial_cost", "median"),
    )
    .sort_values("record_count", ascending=False)
)
job_type_summary


**Discuss:** which borough has the most records? Is it also the borough with the highest median estimated cost? How do the `A1` and `A2` medians compare? Read each median alongside its record count.

The borough summary pools all job types together; the job-type summary pools all boroughs. These are separate comparisons of the same file. Next, we will add a building attribute and use the same grouping idea again.

### 6.2 Inspect the second table and choose a key

**BIN** means Building Identification Number. It is already present as `bin_num` in the cleaned DOB column names. NYC uses the same identifier in its building catalog; street addresses are unnecessary for this join.

The CSV has one row per BIN in this extract. `construction_year` is the catalog's reported year the building was completed. A blank means the year is unavailable; it does not mean the building has no construction year. The extract converts the source's zero/blank unknown codes to missing values.

[NYC field definitions](https://github.com/CityOfNewYork/nyc-geo-metadata/blob/master/Metadata/Metadata_BuildingFootprints.md).


In [ ]:
buildings_df = pd.read_csv(
    "data/raw/nyc_building_attributes_20260927.csv",
    dtype={"bin_num": "string", "construction_year": "Int64"},
)
buildings_df.head()


Use the same dtype for the key in both tables. BIN is an identifier, so we use strings rather than perform arithmetic on it. `Int64` with a capital `I` is pandas' nullable integer dtype; it allows a missing construction year.

**Predict:** should the merged table have one row per building or one row per DOB record?


In [ ]:
permits_work["bin_num"] = permits_work["bin_num"].astype("string")

assert permits_work["bin_num"].notna().all()
assert buildings_df["bin_num"].notna().all()
assert buildings_df["bin_num"].is_unique, "Investigate repeated building IDs before joining."

print("DOB rows:", len(permits_work))
print("Distinct DOB building IDs:", permits_work["bin_num"].nunique())
print("Rows in building extract:", len(buildings_df))
print("Known construction years:", buildings_df["construction_year"].notna().sum())


### 6.3 Preserve the DOB records when joining

Several DOB rows can refer to the same building. This is a **many-to-one** join: many rows on the left may match one building row on the right.

| Argument | Why we use it |
|---|---|
| `on="bin_num"` | Match building IDs |
| `how="left"` | Retain DOB rows even if a building cannot be found |
| `validate="many_to_one"` | Require unique building IDs on the right; prevent accidental multiplication of DOB rows |
| `indicator=True` | Add `_merge` to distinguish matches from unmatched rows |

An inner join would discard unmatched DOB rows. A missing construction year alone cannot tell us whether the building failed to match: the building may match but have an unknown year. Inspect `_merge` for that distinction.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 4: Attach Building Construction Years

Join `permits_work` (left) to `buildings_df` (right) using `bin_num`, keeping every DOB row. Name the result `permits_with_buildings`.

The output should still contain 19,590 rows, including repeats in the original view. The new construction year should agree for every occurrence of the same matched building ID.


In [ ]:
permits_with_buildings = permits_work.merge(
    ...,
    on=...,
    how=...,
    validate="many_to_one",
    indicator=True,
)
permits_with_buildings[["job_num", "bin_num", "initial_cost", "construction_year", "_merge"]].head()


In [ ]:
grader.check("ca06_merge")

<details>
<summary>Try the join first, then reveal a worked solution if needed</summary>

```python
permits_with_buildings = permits_work.merge(
    buildings_df,
    on="bin_num",
    how="left",
    validate="many_to_one",
    indicator=True,
)
```

The left table determines the rows we retain. Multiple records can share one building ID, so `one_to_one` would be the wrong validation here. A match attaches the building's year to each of its records; it does not collapse those records into one job or building.

</details>


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 5: Separate Two Reasons for Missing Years

Calculate three counts:

1. `matched_rows`: rows where `_merge` is `"both"`.
2. `unmatched_rows`: rows where `_merge` is `"left_only"`.
3. `matched_missing_year`: rows that matched a building **and** have a missing `construction_year`.

`.eq(value)` means `== value`. Use `.isna()` to find missing years and `&` to require both conditions. Put each comparison in parentheses, then use `.sum()` to count the `True` results. Do not replace missing years with zero or interpret an unmatched building as demolished.


In [ ]:
matched_rows = ...
unmatched_rows = ...
matched_missing_year = ...

print("Matched DOB rows:", matched_rows)
print("Unmatched DOB rows:", unmatched_rows)
print("Matched, but year unknown:", matched_missing_year)


In [ ]:
grader.check("ca06_matchaudit")

### 6.4 State which records can answer the decade question

We can group by construction decade only when a construction year is available. Keep those rows in a separate DataFrame; retain the complete joined table for auditing.

This snapshot matches 19,488 DOB rows. Of those, 67 have no reported construction year. Another 102 rows do not match the building table. None of those 169 rows can be placed in a construction decade from these sources.

The building catalog is a current snapshot joined to records spanning many years. We use the catalog’s reported construction decade as a grouping label; we do not calculate a building’s age at the time of a historical permit from this join.


In [ ]:
known_year_records = permits_with_buildings.loc[
    permits_with_buildings["construction_year"].notna()
].copy()

# Integer division puts 1927 in the 1920s, 2004 in the 2000s, etc.
known_year_records["construction_decade"] = (
    known_year_records["construction_year"] // 10 * 10
)

assert len(known_year_records) == matched_rows - matched_missing_year
print("Rows available for decade summaries:", len(known_year_records))
known_year_records[["bin_num", "construction_year", "construction_decade", "initial_cost"]].head()


### Summarize a group with more than one measure

`.groupby("construction_decade")` forms the groups. Named aggregation lets us request several summaries together:

```python
table.groupby("group_column").agg(
    output_name=("input_column", "summary_method"),
)
```

Use `"size"` for the number of rows, `"nunique"` for the number of distinct IDs, and `"median"` for the middle cost after sorting (the average of the middle two when there is an even number). The resulting median describes **rows**, including repeated visible rows; it is not a median across unique buildings or jobs. In the supplied chain, `.reset_index()` makes the decade a regular column again, and `.sort_values()` puts the decades in order.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 6: Use the Added Building Information

Create `decade_summary` with one row per `construction_decade` and three summaries:

- `record_count`: number of DOB rows in the group.
- `building_count`: number of distinct `bin_num` values.
- `median_initial_cost`: median of `initial_cost`.

The code supplies grouping and formatting; fill in the three summary methods. Then identify the decade with the most records and compare its record and building counts. Compare the median costs for the 1920s and 1930s: which is higher, and by how many dollars? **Construction decade is the building's reported construction decade, not the date of the alteration filing.**


In [ ]:
decade_summary = (
    known_year_records.groupby("construction_decade")
    .agg(
        record_count=("bin_num", ...),
        building_count=("bin_num", ...),
        median_initial_cost=("initial_cost", ...),
    )
    .reset_index()
    .sort_values("construction_decade")
)
decade_summary


In [ ]:
grader.check("ca06_decades")

### Read the result before making a claim

The chart counts records by the building's reported construction decade. It is **not** a history of construction activity. Use `building_count` to see how many distinct buildings contribute to each bar.

Cost estimates come from different filing years and have not been adjusted for changes in prices over time (inflation). Small groups and repeated records can affect medians. The summary describes this selected file and the buildings we could match; it does not establish that building age causes higher or lower costs.


In [ ]:
import matplotlib.pyplot as plt

BERKELEY_BLUE = "#003262"
BERKELEY_GOLD = "#fdb515"

ax = decade_summary.plot.bar(
    x="construction_decade", y="record_count", color=BERKELEY_BLUE,
    legend=False, figsize=(11, 4.5),
)
ax.set_title("Selected DOB records by the building's reported construction decade", loc="left")
ax.set_xlabel("Reported building construction decade")
ax.set_ylabel("DOB records with a matched construction year")
ax.tick_params(axis="x", labelrotation=45)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 7: Write a Defensible Finding

Write a short paragraph covering these comparisons:

1. Report one comparison from `borough_summary` or `job_type_summary`, naming the groups and their record counts or median dollar amounts.
2. Identify the decade with the most records. Report its record count, distinct building count, and median recorded cost. Include your 1920s–1930s median cost comparison.
3. Explain why the record and building counts differ, and which comparison needed the merge.
4. Name a limitation that prevents this from being a claim about all NYC alteration work or about the causal effect of building age on cost.

Discuss your answer with a partner. This reflection has no automatic correctness check.


_Type your answer here, replacing this text._

<!-- END QUESTION -->

### Return to the checklist

Point to one result or decision for each of the six steps. Then discuss:

- Why did we preserve the repeated visible rows?
- What is a row before the join, after the join, and after grouping by decade?
- How is an unmatched building different from a matched building with an unknown year?
- Which comparisons used only the DOB table, and which required the second dataset?

**The core activity ends here.** The optional practice below reuses these variables.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Optional Practice

<hr style="border: 1px solid #fdb515;" />

Run Steps 1–6 first. Choose an extension if time permits; none is submitted.


### A. Compare join choices using the actual unmatched records

Predict how many rows an inner join will keep. Will the 67 matched rows with unknown years disappear too?


In [ ]:
inner_demo = permits_work.merge(
    buildings_df, on="bin_num", how="inner", validate="many_to_one",
)
print("Left-join rows:", len(permits_with_buildings))
print("Inner-join rows:", len(inner_demo))
print("Unknown years still present:", inner_demo["construction_year"].isna().sum())


<details>
<summary>Compare your prediction</summary>

The left join retains 19,590 rows; the inner join keeps 19,488. It removes the 102 unmatched rows, but retains the 67 rows that match a building with an unknown year. Joining is about matching keys, not about requiring every attribute to be present.

</details>


For a separate demonstration, deliberately duplicate one building row in a copy of the right-hand table. `validate="many_to_one"` should reject it before repeated keys can multiply DOB rows. The original building table remains unchanged.


In [ ]:
buildings_duplicate_demo = pd.concat(
    [buildings_df, buildings_df.iloc[[0]]], ignore_index=True,
)
try:
    permits_work.merge(
        buildings_duplicate_demo, on="bin_num", how="left", validate="many_to_one",
    )
except pd.errors.MergeError as error:
    print("Expected validation error:", error)


### B. Investigate a geographic edge case

A **community board** is a local advisory body serving a community district. This field combines a borough code and a two-digit board number: `208`, for example, encodes the Bronx (`2`) and board `08`. Integer division by 100 extracts that leading borough code. The first digit of `community_board` usually agrees with the borough recorded in the other column. Investigate disagreements before treating them as errors.


In [ ]:
borough_code = permits_work["borough"].map({
    "MANHATTAN": 1,
    "BRONX": 2,
    "BROOKLYN": 3,
    "QUEENS": 4,
    "STATEN ISLAND": 5,
})

community_board_code = (
    permits_work["community_board"]
    .astype("Int64")
    // 100
)

permits_work.loc[
    borough_code.ne(community_board_code),
    ["borough", "community_board", "job_num", "house_num", "street_name"],
]


The check returns **two rows**: both say `MANHATTAN` but have Community Board `208`, and both are on **Marble Hill Avenue**.

Do not automatically "fix" them. NYC Bronx Community Board 8 explicitly includes **Marble Hill**. This is a useful reminder that geographic classifications can encode different administrative realities.

> A failed sanity check can reveal an edge case in the world, not an error in the data.

[NYC Bronx Community Board 8](https://www.nyc.gov/site/communityboards/about/bronx-boards.page)


### C. Select an explicit time window

For a classroom illustration of recorded counts around the 2020–2021 filing-system transition, use **2013–2023**, inclusive. This is a teaching window, not a uniquely correct planning period; it deliberately spans a change in data coverage.

Count what the choice excludes. Rows with no fully-permitted date cannot be placed on this timeline, but were valid inputs to the borough composition exercise.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 8 (Optional): Account for Missing Dates

Calculate `num_missing` and `pct_missing` for `permit_date` in the full `permits_work` table. Express the percentage on a 0–100 scale; rounding to one decimal place is fine. A missing date does not tell us whether a job is pending.


In [ ]:
missing_date_mask = permits_work["permit_date"].isna()
num_missing = ...
pct_missing = ...
print(num_missing, pct_missing)


In [ ]:
grader.check("ca06_missingdates")

In [ ]:
window_accounting = pd.Series({
    "No fully-permitted date": permits_work["permit_date"].isna().sum(),
    "Before 2013": permits_work["perm_year"].lt(2013).sum(),
    "2013–2023 inclusive": permits_work["perm_year"].between(2013, 2023).sum(),
    "After 2023": permits_work["perm_year"].gt(2023).sum(),
}, name="record_count")
assert window_accounting.sum() == len(permits_work)
window_accounting


Combine Boolean conditions with `&` and put each condition in parentheses. The filter excludes missing years as well as years outside the window.


In [ ]:
analysis_period = permits_work.loc[
    (permits_work["perm_year"] >= 2013)
    & (permits_work["perm_year"] <= 2023)
].copy()
print("Rows in the time window:", len(analysis_period))
print("Share of dated rows retained:",
      round(len(analysis_period) / permits_work["permit_date"].notna().sum() * 100, 1), "%")


### A filing-system change complicates the trend

**BIS (Buildings Information System)** is the older filing system; **DOB NOW** is the newer online system. The parent [DOB Job Application Filings dataset](https://data.cityofnewyork.us/Housing-Development/DOB-Job-Application-Filings/ic3t-wcy2) excludes DOB NOW submissions. General Construction alteration filings moved into DOB NOW beginning [December 28, 2020](https://www.nyc.gov/assets/buildings/newsletters/DOB_BN_122120.html); New Building and Alteration-CO followed on [March 1, 2021](https://www.nyc.gov/assets/buildings/newsletters/DOB_BN_022621.html).

The transition had exceptions, and [existing filings could continue in BIS](https://www.nyc.gov/site/buildings/industry/dob-now-build-faqs.page). These are filing-system dates, while our timeline uses **fully-permitted dates**, which can occur later. We cannot assign one universal cutoff date to every A1/A2/A3 record or attribute the whole decline to the transition.

**Hand - DD7: Changes with Time:** a drop in this legacy view can reflect changing coverage as well as changing activity. The view alone cannot isolate the effects of COVID-19 or measure citywide construction.


### Count every calendar year in the window

After counting, `reindex()` adds years with no rows in this selected view. Zero here means **zero observed records in the file**, not zero construction in the city.


In [ ]:
yearly_permits = (
    analysis_period["perm_year"]
    .value_counts()
    .reindex(range(2013, 2024), fill_value=0)
    .rename_axis("year")
    .reset_index(name="count")
)
yearly_permits


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(yearly_permits["year"], yearly_permits["count"], color=BERKELEY_BLUE)
ax.axvline(2021, color=BERKELEY_GOLD, linewidth=2,
           label="2020–21 DOB NOW expansion (context)")
ax.set_title("Records in the selected view by fully-permitted year", loc="left")
ax.set_xlabel("Fully-permitted year (not filing year)")
ax.set_ylabel("Recorded rows")
ax.set_xticks(yearly_permits["year"])
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 9 (Optional): Measure Recorded Change

Copy `yearly_permits` into `yearly_permit_trend` and add `pct_change`: percent change in `count` from the preceding year. Use `.pct_change(fill_method=None) * 100` and round to one decimal place.

Percent change is `(current - previous) / previous * 100`; `.pct_change()` returns the fraction before multiplying by 100. Compare the counts in 2019, 2020, and 2021 with 2018 and 2022, then read the percent changes for 2020 and 2021. Describe the pattern in this file and explain why the filing-system change limits a citywide interpretation. The first year has no previous year for comparison, so its percent change should be missing. If a previous year's count were zero, a percent growth rate would be undefined.


In [ ]:
yearly_permit_trend = yearly_permits.copy()
yearly_permit_trend["pct_change"] = ...
yearly_permit_trend


In [ ]:
grader.check("ca06_pctchange")